# Simple Rubin Injection Demo

This notebook walks through the **minimum end-to-end workflow** for the
star-cluster injection pipeline using a real Rubin (DP0.2) coadd.

You will:

1. Set up the project path and imports
2. Configure the synthetic cluster run (magnitude / size / count / band)
3. Load one Rubin coadd cutout
4. Inject synthetic star clusters into that image
5. Plug in a detector (here: a built-in matched-filter + MCI example)
6. Match detections to the truth catalog and compute completeness
7. Make and save standard or poster-style plots

User-tunable controls live in cells 4 and 13 (display vs. save behavior).

In [ ]:
import os
import sys
import inspect
from pathlib import Path

repo_root = Path('/home/snair63/WORK/INJECT/star-cluster-injection-pipeline').resolve()

os.chdir(repo_root)

if not (repo_root / 'src' / 'config.py').is_file():
    raise RuntimeError(f'src/config.py not found under {repo_root}')

sys.path = [p for p in sys.path if p != str(repo_root)]
sys.path.insert(0, str(repo_root))

for name in list(sys.modules.keys()):
    if name == 'src' or name.startswith('src.'):
        del sys.modules[name]

from lsst.daf.butler import Butler
from src.config import InjectionConfig, ClusterConfig
from src.pipeline import InjectionPipeline

print('cwd now:', os.getcwd())
print('Using repo_root:', repo_root)
print('InjectionConfig loaded from:', inspect.getsourcefile(InjectionConfig))
print('InjectionConfig args:', InjectionConfig.__init__.__code__.co_varnames)

## 1. Configure The Run

This is where most users will edit values.

**Common knobs:**

- `band`: which Rubin filter to load (e.g. `'i'`, `'r'`, `'g'`)
- `cutout_size`: pixel size of the square coadd cutout to load
- `n_clusters`: how many synthetic clusters to inject
- `seed`: reproducibility seed
- `psf_fwhm_fallback`: PSF FWHM to use if the coadd PSF object is unavailable
- `output_dir`: where catalogs/checkpoints are saved (you can also override the **plot** output directory later in cell 13)

**Cluster parameter ranges (`ClusterConfig`):**

- `profile_type`: `'king'`, `'plummer'`, `'eff'`, or `'sersic'`
- `mag_min` / `mag_max`: injected total magnitude range
- `r_half_min` / `r_half_max`: half-light radius range in pixels
- `concentration_min` / `concentration_max`: profile concentration parameter
  (King c, EFF gamma, or Sersic n depending on `profile_type`)

If you ever see an `unexpected keyword argument` error, rerun the import cell above first; it force-reloads local `src` modules.

In [ ]:
BUTLER_REPO = 'dp02'
BUTLER_COLLECTIONS = '2.2i/runs/DP0.2'
DATA_ID = {'tract': 3828, 'patch': 24}

# Safety check: confirms this class supports the notebook arguments.
assert 'cutout_size' in InjectionConfig.__init__.__code__.co_varnames, (
    'Loaded InjectionConfig does not include cutout_size. '
    'Rerun the import cell above (or restart kernel and rerun from top).'
)

config = InjectionConfig(
    run_name='simple_rubin_mci_demo',
    band='i',
    cutout_size=1200,
    n_clusters=250,
    seed=42,
    edge_buffer=50,
    use_actual_psf=True,
    psf_fwhm_fallback=3.5,
    output_dir=str(repo_root / 'plots' / 'simple_rubin_mci_demo'),
    cluster_config=ClusterConfig(
        profile_type='king',
        mag_min=20.0,
        mag_max=26.0,
        r_half_min=2.0,
        r_half_max=10.0,
        concentration_min=5.0,
        concentration_max=30.0,
    ),
)

config

## 2. Load One Rubin Coadd

This pulls a Butler `deepCoadd` for the specified `(tract, patch, band)`,
extracts a square cutout of size `cutout_size`, and stores the matching PSF
object inside the pipeline so PSF-aware injection works automatically.

The next cell shows the loaded image so you can verify it visually.

**Tip:** if you want to test on a synthetic image instead, you can do
`pipe.load_data(image=numpy_array)` and skip the Butler entirely.

In [ ]:
butler = Butler(BUTLER_REPO, collections=BUTLER_COLLECTIONS)
pipe = InjectionPipeline(config)
pipe.load_data(butler=butler, data_id=DATA_ID)

print(f'Loaded image shape: {pipe.image.shape}')
print(f'Active band: {config.band}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

vmin, vmax = np.percentile(pipe.image, [1, 99])
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(pipe.image, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
ax.set_title(f'Loaded {config.band}-band coadd ({pipe.image.shape[1]} x {pipe.image.shape[0]} px)')
ax.set_xlabel('X [pix]')
ax.set_ylabel('Y [pix]')
plt.show()

## 3. Inject Synthetic Clusters

`pipe.generate_catalog()` builds a randomized truth catalog from `ClusterConfig`.
`pipe.inject()` then renders each cluster into the image with the real Rubin PSF
applied at its position and adds Poisson-style noise.

After this cell:
- `pipe.image` is the original image
- `pipe.injected_image` is the image with clusters added
- `pipe.injection_info` holds the per-cluster truth records (x, y, mag, r_half, PSF FWHM, ...)

In [ ]:
catalog = pipe.generate_catalog()
injected_image, injection_info = pipe.inject(
    psf_fwhm_fallback=config.psf_fwhm_fallback,
    verbose=True,
)

print(f'Injected clusters: {len(injection_info)}')

## 4. Detection Function (MCI Example)

The pipeline does not lock you into one detector. Instead, you pass any
callable that takes an image and returns detections.

The example below uses the project's built-in matched-filter + MCI
(Morphological Cluster Index) detector with multi-scale templates.
You can tune the knobs or replace the whole function — see cell 18 for
the minimal pattern.

**Required output format from any detector:**

- `list[dict]`
- each dict must include `x` and `y` pixel coordinates
- optional fields like `flux`, `snr`, `r_half`, `magnitude` unlock extra
  diagnostics later

In [ ]:
from src.detection import run_cluster_detection


def mci_detector(image):
    """Example detector users can replace with their own pipeline."""
    return run_cluster_detection(
        image=image,
        psf_fwhm=config.psf_fwhm_fallback,
        threshold_sigma=3.0,
        mci_max=0.85,
        snr_min=3.0,
        r_half_min=1.0,
        ellipticity_max=0.6,
        box_size=64,
        pixel_scale=config.pixel_scale,
        use_multiscale=True,
        use_mci=True,
        verbose=True,
    )


detections = pipe.detect_with(mci_detector)
print(f'Detections: {len(detections)}')

## 5. Match Detections And Compute Completeness

`pipe.analyze(match_radius=...)` performs nearest-neighbor matching between
truth (`pipe.injection_info`) and detections (`pipe.detection_catalog`)
within `match_radius` pixels, builds a `ClusterRetrieval` object, and
returns summary statistics:

- overall recovery fraction
- 50% magnitude limit
- 50% half-light radius limit
- per-bin completeness numbers

Increase `match_radius` if your detector is positionally noisier.

In [ ]:
stats = pipe.analyze(match_radius=5.0)
stats

## 6. Display And Save Settings

Set these once and the plotting cells below will use them.

- `SHOW_PLOTS_INTERACTIVE`:
  - `True`  → render plots inline in the notebook **and** save to disk
  - `False` → only save plots to disk (faster, useful for batch runs)
- `PLOT_OUTPUT_DIR`: directory for all saved figures
  (defaults to `<output_dir>/plots` from your `InjectionConfig`)
- `PLOTS_TO_MAKE`: which plots to produce

Available plot keys:

`'injection_summary'`, `'before_after'`, `'position_map'`,
`'completeness_1d'`, `'completeness_2d'`, `'psf_fwhm_hist'`, `'psf_fwhm_map'`

**Note:** postage stamp triptychs are always saved automatically when
the data is available — you do not need to opt in.

In [ ]:
from pathlib import Path

# ---- USER CONTROLS ---------------------------------------------------
SHOW_PLOTS_INTERACTIVE = True   # True = display inline + save; False = save only
PLOT_OUTPUT_DIR        = str(Path(config.output_dir) / 'plots')   # change to anywhere on disk

# Plots to display (everything is saved to disk regardless).
PLOTS_TO_MAKE = [
    'injection_summary',   # original + injected + diff + mag/size/flux distributions
    'before_after',        # 3-panel original / injected / difference
    'position_map',        # detected vs missed cluster positions
    'completeness_1d',     # completeness vs magnitude AND vs r_half
    'completeness_2d',     # 2D completeness heatmap (mag x r_half)
    'psf_fwhm_hist',       # PSF FWHM at each injection position
    # 'psf_fwhm_map',      # only available if you pass psf_grid_fwhm=...
]

# How many postage stamp triptychs to save (one PNG each).
# Set to len(pipe.injection_info) to save ALL injected clusters.
N_STAMPS_TO_SAVE = len(pipe.injection_info)
# ----------------------------------------------------------------------

Path(PLOT_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print('Plots will be saved to:', PLOT_OUTPUT_DIR)
print('Interactive display:', SHOW_PLOTS_INTERACTIVE)
print('Postage stamps to save:', N_STAMPS_TO_SAVE)

## 7. Make Plots

This single call produces every plot listed in `PLOTS_TO_MAKE`,
saves them to `PLOT_OUTPUT_DIR`, and (if `SHOW_PLOTS_INTERACTIVE`
is `True`) also displays them inline.

In [ ]:
results = pipe.make_plots(
    output_dir=PLOT_OUTPUT_DIR,
    plots=PLOTS_TO_MAKE,
    show=SHOW_PLOTS_INTERACTIVE,
    save=True,
    n_stamps=N_STAMPS_TO_SAVE,
)

print('Saved files:')
for k, v in results['saved'].items():
    print(f' - {k}: {v}')

## 8. Optional: Plug In Your Own Detector

The cleanest way to use this pipeline is to keep the rest of the workflow
unchanged and only replace the detection function.

Your detector just needs to:
1. Take an image array as input
2. Return a `list[dict]` where each dict has at least `x` and `y`

Then call `pipe.detect_with(my_detector)`, `pipe.analyze(...)`, and
`pipe.make_plots(...)` exactly as above.

In [ ]:
# Example pattern:
# def my_detector(image):
#     return [
#         {'x': 100.0, 'y': 120.0, 'flux': 42.0, 'snr': 5.1},
#     ]
#
# detections = pipe.detect_with(my_detector)
# stats = pipe.analyze(match_radius=5.0)
# results = pipe.make_plots(plots=['completeness_1d'], show=True, save=True)